
# Groningen earthquake forecasting on Databricks

A medallion pipeline (bronze, silver, gold Delta tables) for the Groningen
forecasting project (https://github.com/RiteshDas2000/GroningenSeis):

1. **Bronze**: load the raw KNMI catalogue and the gas production file as they are.
2. **Silver**: clean, type and deduplicate them, and spread the yearly gas volumes over months.
3. **Data quality**: run automated checks and store the results in a Delta table.
4. **Gold**: build the monthly feature table with Spark SQL (calendar spine, joins, `LAG` and rolling window functions).
5. **Models**: train the baseline and the Poisson regression on chronological splits and compare them.

**Before running**: upload `knmi_groningen_events.txt` and `groningen_production_gasyear.csv`
(from the repo's `data/` folder) to the volume created in the next cell.

In [0]:
CATALOG = "workspace"          # the default catalog in Databricks Free Edition
SCHEMA = "groningen"
VOLUME = "raw"

MIN_MAG = 1.5                  # count events with magnitude >= this
START, END = "1995-01-01", "2026-09-01"
TRAIN_END, VAL_END = "2022-12-01", "2024-12-01"

LOCAL_TEST = False             # True only when testing outside Databricks
RAW = "data" if LOCAL_TEST else f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

if LOCAL_TEST:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[2]").getOrCreate()
    def table(name): return name
    def save(df, name): df.createOrReplaceTempView(name)
else:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
    def table(name): return f"{CATALOG}.{SCHEMA}.{name}"
    def save(df, name):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table(name))

spark.conf.set("spark.sql.session.timeZone", "UTC")

## 1. Bronze: raw data, unchanged

In [0]:
from pyspark.sql import functions as F

events_raw = (spark.read.option("header", True).option("sep", "|")
              .csv(f"{RAW}/knmi_groningen_events.txt"))
events_raw = events_raw.toDF(*[c.lstrip("#") for c in events_raw.columns])
save(events_raw.withColumn("_loaded_at", F.current_timestamp()), "bronze_events")

prod_raw = spark.read.option("header", True).csv(f"{RAW}/groningen_production_gasyear.csv")
prod_raw = prod_raw.toDF("gas_year_label", "volume_nm3")      # Dutch headers -> English
save(prod_raw.withColumn("_loaded_at", F.current_timestamp()), "bronze_production")

print(events_raw.count(), "catalogue rows;", prod_raw.count(), "gas years")

1838 catalogue rows; 54 gas years


## 2. Silver: cleaned, typed and deduplicated

In [0]:
silver_events = spark.sql(f"""
    WITH typed AS (
        SELECT EventID                                  AS event_id,
               CAST(Time AS TIMESTAMP)                  AS event_time,
               CAST(Latitude AS DOUBLE)                 AS latitude,
               CAST(Longitude AS DOUBLE)                AS longitude,
               CAST(`Depth/km` AS DOUBLE)               AS depth_km,
               CAST(Magnitude AS DOUBLE)                AS magnitude,
               EventLocationName                        AS location,
               ROW_NUMBER() OVER (PARTITION BY EventID ORDER BY _loaded_at DESC) AS rn
        FROM {table('bronze_events')}
        WHERE EventType = 'induced or triggered event'
    )
    SELECT event_id, event_time, latitude, longitude, depth_km, magnitude, location
    FROM typed WHERE rn = 1                               -- one row per event
""")
save(silver_events, "silver_events")

# Gas years run October-September ("'12 - '13" = Oct 2012 - Sep 2013).
# Only yearly totals exist, so each year is spread evenly over its 12 months (bcm per month).
silver_production = spark.sql(f"""
    WITH years AS (
        SELECT CAST(TRIM(BOTH "' " FROM SPLIT(gas_year_label, '-')[0]) AS INT) AS yy,
               CAST(volume_nm3 AS DOUBLE) AS volume_nm3
        FROM {table('bronze_production')}
    ),
    starts AS (
        SELECT MAKE_DATE(IF(yy >= 50, 1900 + yy, 2000 + yy), 10, 1) AS gas_year_start, volume_nm3
        FROM years
    )
    SELECT EXPLODE(SEQUENCE(gas_year_start, ADD_MONTHS(gas_year_start, 11), INTERVAL 1 MONTH)) AS month,
           volume_nm3 * 1000 / 1e9 / 12 AS production_bcm,
           gas_year_start
    FROM starts
""")
save(silver_production, "silver_production")

## 3. Data quality checks
Each check counts the rows that break a rule. The results are stored as a Delta table,
and the notebook stops if a critical check fails, so bad data never reaches the model.

In [0]:
checks = [
    # (check, severity, SQL returning the number of failing rows)
    ("event_id is unique", "critical",
     f"SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM {table('silver_events')}"),
    ("no missing time or magnitude", "critical",
     f"SELECT COUNT(*) FROM {table('silver_events')} WHERE event_time IS NULL OR magnitude IS NULL"),
    ("magnitude between -1 and 4", "critical",
     f"SELECT COUNT(*) FROM {table('silver_events')} WHERE magnitude NOT BETWEEN -1 AND 4"),
    ("location inside the Groningen region", "warning",
     f"SELECT COUNT(*) FROM {table('silver_events')} WHERE latitude NOT BETWEEN 52.8 AND 53.7 OR longitude NOT BETWEEN 6.0 AND 7.4"),
    ("no events dated in the future", "critical",
     f"SELECT COUNT(*) FROM {table('silver_events')} WHERE event_time > CURRENT_TIMESTAMP()"),
    ("production is non-negative", "critical",
     f"SELECT COUNT(*) FROM {table('silver_production')} WHERE production_bcm IS NULL OR production_bcm < 0"),
    ("one production row per month", "critical",
     f"SELECT COUNT(*) - COUNT(DISTINCT month) FROM {table('silver_production')}"),
    ("no missing gas years", "critical",
     f"""SELECT COUNT(*) FROM (
            SELECT gas_year_start, LAG(gas_year_start) OVER (ORDER BY gas_year_start) AS prev
            FROM (SELECT DISTINCT gas_year_start FROM {table('silver_production')}))
         WHERE prev IS NOT NULL AND gas_year_start <> ADD_MONTHS(prev, 12)"""),
]

rows = [(name, severity, int(spark.sql(sql).first()[0])) for name, severity, sql in checks]
dq = (spark.createDataFrame(rows, "check STRING, severity STRING, failing_rows LONG")
      .withColumn("passed", F.col("failing_rows") == 0)
      .withColumn("checked_at", F.current_timestamp()))
save(dq, "dq_results")
dq.show(truncate=False)

failed = [r for r in rows if r[1] == "critical" and r[2] > 0]
assert not failed, f"Critical data quality checks failed: {failed}"

+------------------------------------+--------+------------+------+--------------------------+
|check                               |severity|failing_rows|passed|checked_at                |
+------------------------------------+--------+------------+------+--------------------------+
|event_id is unique                  |critical|0           |true  |2026-11-17 13:28:06.838991|
|no missing time or magnitude        |critical|0           |true  |2026-11-17 13:28:06.838991|
|magnitude between -1 and 4          |critical|0           |true  |2026-11-17 13:28:06.838991|
|location inside the Groningen region|warning |0           |true  |2026-11-17 13:28:06.838991|
|no events dated in the future       |critical|0           |true  |2026-11-17 13:28:06.838991|
|production is non-negative          |critical|0           |true  |2026-11-17 13:28:06.838991|
|one production row per month        |critical|0           |true  |2026-11-17 13:28:06.838991|
|no missing gas years                |critical|0  

## 4. Gold: monthly feature table
Inputs for month *t*: the counts of the previous 12 months, gas production over the
previous 12 months, and its change from the 12 months before. Target: the count in month *t*.
Only past months are used, so no future data leaks into the features.

In [0]:
lags = ",\n               ".join(f"LAG(n_events, {k}) OVER w AS lag{k}" for k in range(1, 13))

gold = spark.sql(f"""
    WITH calendar AS (            -- every month, so months without events count as 0
        SELECT EXPLODE(SEQUENCE(DATE'{START}', DATE'{END}', INTERVAL 1 MONTH)) AS month
    ),
    counts AS (
        SELECT DATE_TRUNC('MONTH', event_time) AS month, COUNT(*) AS n_events
        FROM {table('silver_events')}
        WHERE magnitude >= {MIN_MAG}
        GROUP BY 1
    ),
    joined AS (
        SELECT c.month,
               CAST(COALESCE(n.n_events, 0) AS DOUBLE) AS n_events,
               COALESCE(p.production_bcm, 0.0)         AS production_bcm   -- 0 after the field closed
        FROM calendar c
        LEFT JOIN counts n            ON n.month = c.month
        LEFT JOIN {table('silver_production')} p ON p.month = c.month
    ),
    features AS (
        SELECT month,
               {lags},
               SUM(production_bcm) OVER (ORDER BY month ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING) AS prod_12m,
               COUNT(production_bcm) OVER (ORDER BY month ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING) AS n_prod,
               n_events AS target
        FROM joined
        WINDOW w AS (ORDER BY month)
    )
    SELECT month, {", ".join(f"lag{k}" for k in range(1, 13))}, prod_12m,
           prod_12m - LAG(prod_12m, 12) OVER (ORDER BY month) AS prod_change,
           target,
           CASE WHEN month <= DATE'{TRAIN_END}' THEN 'train'
                WHEN month <= DATE'{VAL_END}'   THEN 'validation'
                ELSE 'test' END AS split
    FROM features
    WHERE n_prod = 12
""").dropna()
save(gold, "gold_features")
display(gold) if not LOCAL_TEST else gold.show(5)

month,lag1,lag2,lag3,lag4,lag5,lag6,lag7,lag8,lag9,lag10,lag11,lag12,prod_12m,prod_change,target,split
1997-01-01,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,40.35190245058521,5.436514747171799,1.0,train
1997-02-01,1.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,39.80505500030667,4.103327619336888,1.0,train
1997-03-01,1.0,1.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,39.25820755002814,2.7701404915019765,0.0,train
1997-04-01,0.0,1.0,1.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,38.711360099749605,1.4369533636670653,0.0,train
1997-05-01,0.0,0.0,1.0,1.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,38.16451264947107,0.10376623583215405,0.0,train
1997-06-01,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,1.0,1.0,0.0,0.0,37.617665199192544,-1.22942089200275,2.0,train
1997-07-01,2.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,1.0,1.0,0.0,37.07081774891401,-2.5626080198376613,0.0,train
1997-08-01,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,1.0,1.0,36.52397029863548,-3.8957951476725654,1.0,train
1997-09-01,1.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,1.0,35.97712284835695,-5.228982275507484,0.0,train
1997-10-01,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,35.43027539807842,-6.562169403342388,0.0,train


## 5. Models: baseline vs Poisson regression on chronological splits

In [0]:
import numpy as np
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler

df = spark.table(table("gold_features")).orderBy("month").toPandas()
lag_cols = [f"lag{k}" for k in range(1, 13)]
X, y = df[lag_cols + ["prod_12m", "prod_change"]], df["target"].to_numpy()
train = (df["split"] == "train").to_numpy()

def mse(a, b): return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

past_avg = X[lag_cols].mean(axis=1)
slope, intercept = np.polyfit(past_avg[train], y[train], 1)
pred_base = intercept + slope * past_avg

Xs = StandardScaler().fit(X[train]).transform(X)        # scale with training data only
pred_poisson = PoissonRegressor(alpha=1.0, max_iter=1000).fit(Xs[train], y[train]).predict(Xs)

results = [(split, model, mse(pred[m], y[m]))
           for split in ["train", "validation", "test"]
           for m in [(df["split"] == split).to_numpy()]
           for model, pred in [("baseline", pred_base), ("Poisson regression", pred_poisson)]]
res = spark.createDataFrame(results, "split STRING, model STRING, mse DOUBLE")
save(res.withColumn("run_at", F.current_timestamp()), "model_results")
res.show(truncate=False)

+----------+------------------+------------------+
|split     |model             |mse               |
+----------+------------------+------------------+
|train     |baseline          |1.4430523854941866|
|train     |Poisson regression|1.3865333655617884|
|validation|baseline          |0.9150054183984561|
|validation|Poisson regression|0.8121054506117907|
|test      |baseline          |0.7291512579667527|
|test      |Poisson regression|0.6957226241540255|
+----------+------------------+------------------+

